In [1]:
! pip install torch

In [2]:
import torch
import torch.nn as nn
import math

In [3]:
# Example sentence
sentence = ["ChatGPT", "is", "great", "at", "explaining", "things", "clearly"]
seq_len = len(sentence)
vocab_size = 10000
embedding_dim = 16
num_heads = 2
hidden_dim = 32

In [4]:
# 1️⃣ Token embedding
token_embedding = nn.Embedding(vocab_size, embedding_dim)
token_ids = torch.arange(seq_len)
tokens_emb = token_embedding(token_ids)  # [seq_len, embedding_dim]
print("Token embeddings:\n", tokens_emb, "\n")

Token embeddings:
 tensor([[ 1.4133,  0.7947, -1.2439,  0.8180,  2.2739, -0.9891,  0.7022,  0.4754,
         -0.6035,  0.1780,  0.0550, -1.0980,  1.3103,  0.8445,  0.9097,  0.8000],
        [-1.1679, -1.2385,  0.4210, -0.6686, -1.5373, -1.4746,  0.0922,  0.9757,
         -0.9359, -0.8018, -1.0100, -0.5766,  0.6241,  0.3849,  0.2815,  0.5510],
        [ 0.9434, -0.5192,  2.3030,  0.6819, -0.6141, -0.2243,  0.9914, -1.5457,
          1.6207,  1.2023,  0.6996, -0.5249,  0.3201,  2.1677,  1.0545, -0.5187],
        [-0.7751,  1.1391,  1.5379, -0.1383,  1.1866, -1.2226, -1.4729, -0.6762,
          0.0249, -0.6465, -0.8198, -1.2245, -0.1033, -1.5134,  0.8653, -1.0475],
        [ 1.9967, -1.1425, -0.1115, -0.4746,  1.5376,  0.2451, -0.2186,  0.1374,
         -0.2260,  2.0587, -0.4053, -0.7463,  0.4836, -0.1134, -0.1784,  1.7483],
        [ 0.8349,  0.5853,  0.2021, -0.9507, -0.8466, -1.2760, -0.3483, -0.5038,
         -0.5645,  0.8750,  0.1136, -0.6148,  0.2569,  1.3988,  0.2981,  0.5759],
   

In [5]:
# 2️⃣ Positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

In [6]:
pos_encoder = PositionalEncoding(embedding_dim)
emb_with_pos = pos_encoder(tokens_emb)  # [seq_len, embedding_dim]
print("Embeddings + positional encoding:\n", emb_with_pos)

Embeddings + positional encoding:
 tensor([[ 1.4133,  1.7947, -1.2439,  1.8180,  2.2739,  0.0109,  0.7022,  1.4754,
         -0.6035,  1.1780,  0.0550, -0.0980,  1.3103,  1.8445,  0.9097,  1.8000],
        [-0.3264, -0.6982,  0.7320,  0.2818, -1.4374, -0.4796,  0.1238,  1.9752,
         -0.9259,  0.1981, -1.0068,  0.4234,  0.6251,  1.3849,  0.2818,  1.5510],
        [ 1.8527, -0.9354,  2.8941,  1.4885, -0.4155,  0.7558,  1.0546, -0.5477,
          1.6407,  2.2021,  0.7059,  0.4751,  0.3221,  3.1677,  1.0551,  0.4813],
        [-0.6340,  0.1491,  2.3505,  0.4445,  1.4821, -0.2673, -1.3782,  0.3193,
          0.0549,  0.3531, -0.8103, -0.2246, -0.1003, -0.5134,  0.8662, -0.0475],
        [ 1.2399, -1.7961,  0.8421, -0.1735,  1.9270,  1.1661, -0.0924,  1.1294,
         -0.1860,  3.0579, -0.3926,  0.2537,  0.4876,  0.8866, -0.1771,  2.7483],
        [-0.1241,  0.8690,  1.2021, -0.9611, -0.3671, -0.3984, -0.1909,  0.4837,
         -0.5145,  1.8737,  0.1294,  0.3850,  0.2619,  2.3988,  0.299

In [7]:
# 3️⃣ Tiny Transformer Encoder layer
encoder_layer = nn.TransformerEncoderLayer(
    d_model=embedding_dim,
    nhead=num_heads,
    dim_feedforward=hidden_dim,
    batch_first=True
)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)

In [8]:
emb_with_pos.shape

torch.Size([7, 16])

In [9]:
# Transformer expects [batch_size, seq_len, embedding_dim], when batch_first=True
# Here batch_size=1
emb_with_pos = emb_with_pos.unsqueeze(0)
emb_with_pos.shape

torch.Size([1, 7, 16])

In [10]:
output = transformer_encoder(emb_with_pos)

print("Transformer output shape:", output.shape)
print("Transformer output:\n", output.squeeze(1))

Transformer output shape: torch.Size([1, 7, 16])
Transformer output:
 tensor([[[ 0.5349,  1.5664, -2.6326,  1.1175,  0.9078, -0.7874, -0.1090,
          -0.0969, -1.2079, -0.2753, -0.2061, -0.3602, -0.4657,  0.8069,
           0.2092,  0.9984],
         [-0.4194,  0.2912, -0.5213,  1.0255, -1.3113, -1.3238, -0.2328,
           0.6594, -0.7106, -1.0598,  0.0175,  0.8538, -0.9507,  1.3098,
           0.0324,  2.3403],
         [ 0.8208, -0.7554,  0.9138,  1.0847, -1.9833, -0.2644, -0.3455,
          -2.0081,  0.4234,  1.1266,  0.5067, -0.4996, -0.5736,  1.5124,
          -0.0696,  0.1110],
         [-0.2240,  1.3671,  1.3838,  0.8548,  1.0660, -1.0139, -1.8984,
           0.8708,  0.0202, -1.3520,  0.1271,  0.0772, -1.5110, -0.6295,
           0.4910,  0.3708],
         [ 0.8408, -2.0631, -0.3286, -0.2205,  0.5907,  0.0127, -0.6386,
          -0.0882, -0.8373,  1.6299, -0.1890,  0.3419, -0.5022,  0.1907,
          -1.0312,  2.2919],
         [-0.1173,  0.9871,  0.2655, -0.8729, -1.2338, 